In [ ]:
from ddi.data import build_human
from ddi.manifest import load_dataset

train, dev, val = build_human()
v14, _ = load_dataset("20260807-123340-ff79db")

In [ ]:
import re, statistics
from collections import Counter

ROLE = re.compile(
    r"\b(previous(ly)?|earlier|discontinu\w+|stopped|withdraw\w+|"
    r"subsequent(ly)?|thereafter|later|after the other\w*|"
    r"comparator|separately|comparison|"
    r"alternative|unsuitable|first choice|instead|"
    r"not given|not permitted|excluded|withheld|not administered)\b", re.I)

def near_marker(text, chars=60):
    """text within `chars` either side of each entity span"""
    out = []
    for k in (1, 2):
        m = re.search(rf"\[E{k}\].*?\[/E{k}\]", text, re.S)
        if m:
            out.append(text[max(0, m.start()-chars):m.end()+chars])
    return " ".join(out)

def role_adjacency(instances, name):
    c = Counter()
    for r in instances:
        hit = bool(ROLE.search(near_marker(r["text"])))
        c[(r["label"] != "NONE", hit)] += 1
    pos_rate = c[(True, True)] / max(c[(True, True)] + c[(True, False)], 1)
    neg_rate = c[(False, True)] / max(c[(False, True)] + c[(False, False)], 1)
    print(f"{name:8s} role language near marker: POS {pos_rate:.3f}  NONE {neg_rate:.3f}"
          f"  ratio {neg_rate / max(pos_rate, 1e-9):.2f}")

role_adjacency(v14, "v14")
role_adjacency(train, "human")

In [ ]:
# P_ROLE=0.2 smoke test
# Change ONE thing in ddi/prompt.py before running: P_ROLE = 0.55 -> 0.2
# Everything else stays as it was for v14-full-2, so the comparison is single-variable.
import os
from openai import OpenAI
import importlib, json, re
from collections import Counter

import ddi.prompt, ddi.resolve, ddi.gates, ddi.synth
for m in (ddi.synth, ddi.prompt, ddi.resolve, ddi.gates):
    importlib.reload(m)

from ddi.data import build_human
from ddi.vocab import build_vocab
from ddi.prompt import make_v14_specs, make_v14_sample_fn, v14_fingerprint, P_ROLE
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi import gates

GEN = "v14-lowrole-check"
V14_ID = "20260807-123340-ff79db"
MODEL, API, EFFORT = "gpt-oss-120b", "responses", "low"

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=60.0)

print(f"P_ROLE = {P_ROLE}  (must be 0.2)")
print(f"prompt sha {v14_fingerprint()}")

vocab = build_vocab()
train, dev, val = build_human()
v14, _ = load_dataset(V14_ID)


# ---- generate -------------------------------------------------------------
specs = make_v14_specs(300, vocab=vocab, seed=0)

n_roled = sum(len(s["roles"]) for s in specs)
n_np = sum(len([e for e in s["entities"]
                if e["key"] not in {k for a in s["asserts"] for k in a["between"]}])
           for s in specs)
print(f"specs: {n_roled}/{n_np} non-participants carry a role "
      f"({n_roled / max(n_np, 1):.2f})")

generate_raw(specs, make_v14_sample_fn(client, model=MODEL, reasoning_effort=EFFORT,
                                       api=API),
             gen_id=GEN, max_workers=16)

did, stats = build_dataset_from_raw(
    GEN, resolver=v14_sample_to_instances, mode="markers",
    generator={"prompt_sha": v14_fingerprint(), "p_role": P_ROLE,
               "model": MODEL, "reasoning_effort": EFFORT,
               "note": "P_ROLE ablation against v14-full-2"},
    vocab_source=vocab.fingerprint(), seed=0)
print(stats["reject_reasons"])
inst, _ = load_dataset(did)


# ---- manipulation check: did the cue actually move? -----------------------
ROLE = re.compile(
    r"\b(previous(ly)?|earlier|discontinu\w+|stopped|withdraw\w+|"
    r"subsequent(ly)?|thereafter|later|after the other\w*|"
    r"comparator|separately|comparison|"
    r"alternative|unsuitable|first choice|instead|"
    r"not given|not permitted|excluded|withheld|not administered)\b", re.I)

def near_marker(text, chars=60):
    out = []
    for k in (1, 2):
        m = re.search(rf"\[E{k}\].*?\[/E{k}\]", text, re.S)
        if m:
            out.append(text[max(0, m.start() - chars):m.end() + chars])
    return " ".join(out)

def role_adjacency(instances, name):
    c = Counter()
    for r in instances:
        c[(r["label"] != "NONE", bool(ROLE.search(near_marker(r["text"]))))] += 1
    pos = c[(True, True)] / max(c[(True, True)] + c[(True, False)], 1)
    neg = c[(False, True)] / max(c[(False, True)] + c[(False, False)], 1)
    print(f"{name:12s} POS {pos:.3f}  NONE {neg:.3f}  ratio {neg / max(pos, 1e-9):.2f}")
    return neg

print("\nrole language near marker")
role_adjacency(train, "human")          # 0.018 / 0.005, ratio 0.27
role_adjacency(v14, "v14 (0.55)")       # 0.228 / 0.468, ratio 2.05
new_neg = role_adjacency(inst, "v14 (0.20)")
print(f"\nNONE adjacency {0.468:.3f} -> {new_neg:.3f}; corpus is 0.005")


# ---- side effects: the other gates must not have moved -------------------
gates.report(inst, records=generation_records(GEN), strict=False)


# ---- read the zero-assert sentences --------------------------------------
# most non-participants now have no role, so the scene has to carry them.
# the failure to look for is bare lists returning.
shown = 0
for line in (RAW / f"{GEN}.jsonl").read_text().splitlines():
    r = json.loads(line)
    if r.get("error") or r["spec"]["asserts"] or shown >= 10:
        continue
    print(f"[{len(r['spec']['entities'])} ents, {len(r['spec']['roles'])} roles] "
          f"{r['sample']['sentence']}")
    shown += 1

In [ ]:
import os
from openai import OpenAI

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=60.0)

In [ ]:
# P_ROLE = 0.2 ablation: full run, train, compare against v14-full-2
#
# Single variable against v14-full-2, with one caveat: P_ROLE also shifted the
# hard-negative rate (0.499 -> 0.612 in smoke), so composition is not perfectly held.
# Record that in the ablation row rather than claiming a clean isolation.
#
# Prerequisite: ddi/prompt.py has P_ROLE = 0.2 and nothing else changed since
# v14-full-2. Commit before running; manifests are landing DIRTY.

# ===========================================================================
# Cell 1: generate. ~30 min at 3.5 req/s. Run under tmux if you want to detach.
# ===========================================================================
import importlib, json, re, statistics, random
from collections import Counter
import pandas as pd

import ddi.prompt, ddi.resolve, ddi.gates, ddi.synth, ddi.train
for m in (ddi.synth, ddi.prompt, ddi.resolve, ddi.gates, ddi.train):
    importlib.reload(m)

from ddi.data import build_human, load_brat_docs
from ddi.vocab import build_vocab
from ddi.prompt import make_v14_specs, make_v14_sample_fn, v14_fingerprint, P_ROLE
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi.train import train_and_eval
from ddi import gates

GEN = "v14-lowrole-full"
V14_ID = "20260807-123340-ff79db"
V13_ID = "<the ~19k v13 dataset id>"
MODEL, API, EFFORT = "gpt-oss-120b", "responses", "low"

assert P_ROLE == 0.2, f"P_ROLE is {P_ROLE}, expected 0.2"
print(f"prompt sha {v14_fingerprint()}")

vocab = build_vocab()
specs = make_v14_specs(6000, vocab=vocab, seed=0)
generate_raw(specs, make_v14_sample_fn(client, model=MODEL, reasoning_effort=EFFORT,
                                       api=API),
             gen_id=GEN, max_workers=16)

low_id, stats = build_dataset_from_raw(
    GEN, resolver=v14_sample_to_instances, mode="markers",
    generator={"prompt_sha": v14_fingerprint(), "p_role": P_ROLE,
               "model": MODEL, "reasoning_effort": EFFORT, "composition": "prior",
               "n_positives_by_k": "hand-tuned to decorrelate positive rate from "
                                   "entity count; not a corpus measurement",
               "note": "P_ROLE ablation against v14-full-2"},
    vocab_source=vocab.fingerprint(), seed=0,
    notes="v14 with P_ROLE=0.2, 6000 specs")
print(low_id, stats["reject_reasons"])



In [ ]:
import importlib, json, re, statistics, random
from collections import Counter
import pandas as pd

import ddi.prompt, ddi.resolve, ddi.gates, ddi.synth, ddi.train
for m in (ddi.synth, ddi.prompt, ddi.resolve, ddi.gates, ddi.train):
    importlib.reload(m)

from ddi.data import build_human, load_brat_docs
from ddi.vocab import build_vocab
from ddi.prompt import make_v14_specs, make_v14_sample_fn, v14_fingerprint, P_ROLE
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi.train import train_and_eval
from ddi import gates

# ===========================================================================
# Cell 2: manipulation check + gates at full scale.
# The probe is scale-dependent, so this number is not comparable to the smoke run.
# ===========================================================================

GEN = "v14-lowrole-full"
V14_ID = "20260807-123340-ff79db"
low_id = "20260812-015921-2b80ca"

train, dev, val = build_human()
v14, _ = load_dataset(V14_ID)
low, _ = load_dataset(low_id)
print(f"human {len(train)}, v14 {len(v14)}, low {len(low)} instances")

ROLE = re.compile(
    r"\b(previous(ly)?|earlier|discontinu\w+|stopped|withdraw\w+|"
    r"subsequent(ly)?|thereafter|later|after the other\w*|"
    r"comparator|separately|comparison|"
    r"alternative|unsuitable|first choice|instead|"
    r"not given|not permitted|excluded|withheld|not administered)\b", re.I)

def near_marker(text, chars=60):
    out = []
    for k in (1, 2):
        m = re.search(rf"\[E{k}\].*?\[/E{k}\]", text, re.S)
        if m:
            out.append(text[max(0, m.start() - chars):m.end() + chars])
    return " ".join(out)

def role_adjacency(instances, name):
    c = Counter()
    for r in instances:
        c[(r["label"] != "NONE", bool(ROLE.search(near_marker(r["text"]))))] += 1
    pos = c[(True, True)] / max(c[(True, True)] + c[(True, False)], 1)
    neg = c[(False, True)] / max(c[(False, True)] + c[(False, False)], 1)
    print(f"{name:12s} POS {pos:.3f}  NONE {neg:.3f}  ratio {neg / max(pos, 1e-9):.2f}")

print("\nrole language near marker")
role_adjacency(train, "human")
role_adjacency(v14, "v14 (0.55)")
role_adjacency(low, "low (0.20)")


In [ ]:

gates.report(low, records=generation_records(GEN), strict=False)



In [ ]:

# ===========================================================================
# Cell 3: train. 3 seeds, ~1 min each. v14 numbers are already known
# (F1 0.379, P 0.302, R 0.511) so only the new arm needs running.
# ===========================================================================
BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
        "neg_ratio": None, "render_mode": "markers"}

rows, preds = [], {}
for seed in [0, 1, 2]:
    cfg = {**BASE, "seed": seed, "dataset": "v14-lowrole"}
    m, p = train_and_eval(cfg, low, dev, return_preds=True)
    if seed == 0:
        preds["low"] = p
    rows.append({"arm": "v14-lowrole", "seed": seed,
                 "f1": m["micro_f1_pos"], "p": m["micro_p_pos"], "r": m["micro_r_pos"],
                 "f1_DrugBank": m.get("micro_f1_pos_DrugBank"),
                 "f1_MedLine": m.get("micro_f1_pos_MedLine")})
    print(f"lowrole seed={seed}  f1={m['micro_f1_pos']:.3f}  "
          f"p={m['micro_p_pos']:.3f}  r={m['micro_r_pos']:.3f}")

df = pd.DataFrame(rows)
print(df[["f1", "p", "r", "f1_DrugBank", "f1_MedLine"]].agg(["mean", "std"]))

print("\nreference (3 seeds, same dev):")
print("  human        F1 0.800  P 0.768  R 0.835")
print("  v13          F1 0.280  P 0.182  R 0.610")
print("  v14 (0.55)   F1 0.379  P 0.302  R 0.511")
print("\nprecision is the target. seed sd ~0.013, so P > ~0.34 is real.")
print("if P rises and R falls proportionally, that is a trade, not a fix.")



In [ ]:

# ===========================================================================
# Cell 4: where the change landed. Only worth running if F1 moved.
# ===========================================================================
from scripts.pair_buckets import sent_entity_counts, bucket_scores, compare

counts = sent_entity_counts(load_brat_docs("Train"))

def fpr_table(rows, name):
    print(f"\n{name}")
    print(f"{'bucket':<8} {'pairs':>6} {'gold+':>6} {'pred+':>6} {'FPR':>7} {'P':>6} {'R':>6}")
    for r in rows:
        fp = r["predicted"] - r["P"] * r["predicted"]
        neg = r["n_pairs"] - r["support"]
        print(f"{r['bucket']:<8} {r['n_pairs']:>6} {r['support']:>6} {r['predicted']:>6} "
              f"{fp / neg if neg else 0:>7.3f} {r['P']:>6.3f} {r['R']:>6.3f}")

low_rows = bucket_scores(dev, preds["low"], counts)
fpr_table(low_rows, "v14-lowrole")
print("\nv14 (0.55) for reference: FPR 0.580 0.466 0.359 0.130 0.051 0.000")

# in-distribution only (2-5 entities), which removes the coverage confound.
# v14 was P 0.315, R 0.668, F1 0.428; human P 0.797, R 0.846, F1 0.821.
sub = [r for r in low_rows if r["bucket"] in ("2-2", "3-3", "4-5")]
tp = sum(r["P"] * r["predicted"] for r in sub)
pp = sum(r["predicted"] for r in sub)
gp = sum(r["support"] for r in sub)
P, R = tp / max(pp, 1), tp / max(gp, 1)
print(f"\nin-distribution (2-5 ents): P {P:.3f}  R {R:.3f}  "
      f"F1 {2 * P * R / max(P + R, 1e-9):.3f}")

from sklearn.metrics import classification_report
print(classification_report([r["label"] for r in dev], preds["low"],
                            labels=["MECHANISM", "EFFECT", "ADVISE", "INT"],
                            zero_division=0))
# v14 per-class precision was MECHANISM 0.23, EFFECT 0.42, ADVISE 0.30, INT 0.22



In [ ]:

# ===========================================================================
# Cell 5: probe at matched sentence count. Only meaningful against a human
# subsample of the same size; full human scores higher purely on volume.
# ===========================================================================
def probe_repeated(instances, n_seeds=10):
    xs = [gates.shortcut_probe(instances, seed=s)[0] for s in range(n_seeds)]
    return statistics.median(xs), statistics.stdev(xs)

n_human_sents = len({r["sent_id"] for r in train})
low_sents = sorted({r["sent_id"] for r in low})

print("human (full)", *[f"{v:.3f}" for v in probe_repeated(train)])
for d in range(3):
    keep = set(random.Random(d).sample(low_sents, min(n_human_sents, len(low_sents))))
    sub = [r for r in low if r["sent_id"] in keep]
    med, sd = probe_repeated(sub)
    print(f"low draw {d}: {len(sub):>6} instances, {med:.3f} (sd {sd:.3f})")
print("\nv14 (0.55) was 0.216, 0.223, 0.228 at matched sentence count; human 0.152")